In [37]:
import json
import pandas as pd
import glob
import seaborn as sns
import re
import os
import matplotlib.pyplot as plt


def gather_results(results_folder, epochs=None, runs=None):
    """Load pretraining results from results/pretraining/seed_N/."""

    all_files = []
    for root, dirs, files in os.walk(results_folder):
        for fname in files:
            if fname.endswith('.json') and 'config' not in fname:
                all_files.append(os.path.join(root, fname))

    all_rows = []
    for f in all_files:
        with open(f) as fh:
            d = json.load(fh)

        # run number from filename: results_1.json -> run=1
        m = re.search(r'results_(\d+)\.json', os.path.basename(f))
        d['run'] = int(m.group(1)) if m else -99

        all_rows.append(d)

    if not all_rows:
        raise ValueError("No results - check settings")

    final_df = pd.DataFrame(all_rows)

    if epochs is not None:
        final_df = final_df[final_df['num_epochs'].isin(epochs)]
    if runs is not None:
        final_df = final_df[final_df['run'].isin(runs)]

    if final_df.empty:
        raise ValueError("No results after filtering - check settings")

    return final_df


def results_barchart(final_df, save_path = None):
    """Bar chart of train_acc and test_acc grouped by number of training epochs."""

    df_melted = final_df.melt(
        id_vars=['num_epochs'],
        value_vars=['train_acc', 'test_acc'],
        var_name='Metric',
        value_name='Accuracy'
    )
    # train_acc -> Train Accuracy, test_acc -> Test Accuracy
    df_melted['Metric'] = df_melted['Metric'].map({'train_acc': 'Train Accuracy', 'test_acc': 'Test Accuracy'})

    plt.clf()
    plt.figure(figsize=(12, 6))
    sns.set_style("whitegrid")

    g = sns.barplot(
        data=df_melted,
        x='num_epochs',
        y='Accuracy',
        hue='Metric',
        errorbar='sd',
        capsize=.1,
        palette=['#1f77b4', '#ff7f0e']
    )

    stats = df_melted.groupby(['num_epochs', 'Metric'])['Accuracy'].agg(['mean', 'std']).reset_index()

    sorted_epochs = sorted(df_melted['num_epochs'].unique())

    hue_order = ['Train Accuracy', 'Test Accuracy'] 

    for i, container in enumerate(g.containers):
        # Map the container index to the correct metric name
        metric = hue_order[i] 
        
        labels = []
        for j, bar in enumerate(container):
            # Get the epoch for this bar's position
            epoch = sorted_epochs[j]
            
            # Filter the stats
            subset = stats[(stats['num_epochs'] == epoch) & (stats['Metric'] == metric)]
            
            if not subset.empty:
                std = subset['std'].values[0]
                height = bar.get_height()
                if pd.isna(std) or std == 0:
                    label = f"{height:.2f}"
                else:
                    label = f"{height:.2f}\n({std:.2f})"
            else:
                label = f"{bar.get_height():.2f}"
            
            labels.append(label)
        
        g.bar_label(container, labels=labels, padding=5, fontsize=10)

    # plt.title("Training and Test Accuracy, by Number of Training Epochs", pad=40)
    plt.xlabel("")
    plt.ylabel("Accuracy")
    plt.legend(title=None, loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=2)
    plt.ylim(0, 110)

    if save_path is None:
        plt.show()
        return
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()

In [38]:
results = gather_results("results/seed_4/pretraining")
# results
results_barchart(results, save_path = "results/seed_4/plots/pretraining_results.svg")

<Figure size 640x480 with 0 Axes>